In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

In [8]:
# text preprocessing

#1.coverting to lowercase

In [9]:
df["review"] = df["review"].str.lower()

#2.Removing the URLs

In [10]:
import re
def remove_urls(text):
    text = re.sub(r"https\S+","",text)
    return text
    
df["review"]=df["review"].apply(remove_urls)

#3.Removing punctuations

In [11]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text
df["review"]=df["review"].apply(remove_punctuations)

#4.Remove html

In [12]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text
df["review"]=df["review"].apply(remove_html)

#5.Removing the Stopwords

In [13]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\INDIA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\INDIA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\INDIA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")
    return text
df["review"]=df["review"].apply(remove_stopwords)

In [16]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


#6.stemming

In [17]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)
df["review"]=df["review"].apply(stemming)

In [18]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


#7.Encoding

In [19]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

y=df["sentiment"]

In [20]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

#8.vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
X=tf.fit_transform(df["review"])

##Dataset & dataloader

In [22]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [23]:
X_train.shape

(39665, 5000)

In [24]:
X_test.shape

(9917, 5000)

In [25]:
import torch
from torch.utils.data import TensorDataset,DataLoader

In [26]:
X_train =X_train.toarray()
X_test =X_test.toarray()

In [27]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [28]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

#Build our RNN

In [29]:
import torch.nn as nn
import torch.optim as optim

In [30]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers =1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first=True)

        #fully connected layer
        self.fc = nn.Linear(hidden_size,1)
    def forward(self,x):
        #optional => shape(num of layers,batch size,hidden size)
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out, _ =self.rnn(x,h0)
        #1st value =>hidden state of all the timesteps =>(batch,seq_len,hidden_Size)
        #2nd value =>final hidden state of last time stamp

        out = self.fc(out[:,-1,:])
        return out

In [31]:
input_size =X_train.shape[1]
model  = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

#Training

In [32]:
#unsqueeze -->one dimesion add additionally
#squeeze -->one dimension reduces

In [33]:
epochs =10
for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1) # add singleton direction
        outputs = model(Xb) #(batch_size,1)
        outputs = torch.sigmoid(outputs.squeeze())#(batch_size,)=>probability

        loss=criterion(outputs,yb)
        loss.backward() #backprop
        optimizer.step()#weights update
    print(f"epoch ={epoch+1}/epochs and loss ={loss.item()}")

epoch =1/epochs and loss =0.3890315592288971
epoch =2/epochs and loss =0.17209294438362122
epoch =3/epochs and loss =0.34815314412117004
epoch =4/epochs and loss =0.1932014673948288
epoch =5/epochs and loss =0.23774120211601257
epoch =6/epochs and loss =0.19427835941314697
epoch =7/epochs and loss =0.18978667259216309
epoch =8/epochs and loss =0.23909719288349152
epoch =9/epochs and loss =0.24400915205478668
epoch =10/epochs and loss =0.2703530788421631


In [34]:
#evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals =0
    for Xb,yb in test_loader:
        Xb=Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()
        tot_vals += yb.size(0)
        correct_vals +=(predicted ==yb).sum().item()
    print(f"accuracy ={correct_vals/tot_vals*100}")

accuracy =85.68115357466975


In [35]:
import pickle
pickle.dump(tf, open("tfidf.pkl", "wb"))

In [36]:
torch.save(model.state_dict(), "rnn_model.pt")

In [37]:
import nltk
nltk.download("punkt")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\INDIA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\INDIA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True